# E3 / E4 — Exploración inicial del dataset y visualización imagen + máscara

**Objetivo del notebook**

Este notebook corresponde a las épicas técnicas **E3 — Gestión de datasets** y **E4 — Preprocesamiento inicial**.

El objetivo es dejar evidencia reproducible de la exploración inicial del dataset sagital: estructura de archivos, lectura de imagen médica, lectura de máscara, inspección de metadatos básicos, visualización de imagen + máscara y exportación de una tabla resumen.

**Alcance**

Este notebook no entrena modelos ni emite diagnósticos. Solo prepara y documenta la base técnica para el motor Python de segmentación.

## 1. Instalación de dependencias

In [ ]:
!pip install -q SimpleITK nibabel pydicom pandas matplotlib scikit-image scipy

## 2. Imports y configuración general

In [ ]:
from __future__ import annotations

import json
import os
import re
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk

## 3. Montaje de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Definición de rutas

Ajustar `PROJECT_ROOT` y `DATASET_ROOT` según dónde esté ubicado el dataset en Google Drive.

Estructura recomendada:

```text
/MyDrive/PFI_MVP/
├── data/
│   └── SPIDER/
├── outputs/
│   ├── figures/
│   ├── metrics/
│   ├── masks/
│   ├── tables/
│   └── reports/
└── docs/
```

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
DATASET_ROOT = PROJECT_ROOT / "data" / "SPIDER"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_ROOT / "figures"
TABLES_DIR = OUTPUT_ROOT / "tables"
MASKS_DIR = OUTPUT_ROOT / "masks"
REPORTS_DIR = OUTPUT_ROOT / "reports"

for directory in [PROJECT_ROOT, DATASET_ROOT, OUTPUT_ROOT, FIGURES_DIR, TABLES_DIR, MASKS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## 5. Búsqueda y listado de archivos

In [ ]:
SUPPORTED_SUFFIXES = (".mha", ".mhd", ".nii", ".nii.gz", ".nrrd", ".dcm")

MASK_KEYWORDS = [
    "mask", "seg", "segmentation", "label", "labels",
    "annotation", "annotations", "gt", "groundtruth", "manual",
]

def has_supported_suffix(path: Path) -> bool:
    name = path.name.lower()
    return any(name.endswith(suffix) for suffix in SUPPORTED_SUFFIXES)

def is_likely_mask(path: Path) -> bool:
    name = path.name.lower()
    return any(keyword in name for keyword in MASK_KEYWORDS)

def file_size_mb(path: Path) -> float:
    try:
        return path.stat().st_size / (1024 ** 2)
    except FileNotFoundError:
        return 0.0

all_files = sorted([p for p in DATASET_ROOT.rglob("*") if p.is_file() and has_supported_suffix(p)])
image_candidates = [p for p in all_files if not is_likely_mask(p)]
mask_candidates = [p for p in all_files if is_likely_mask(p)]

print(f"Archivos médicos encontrados: {len(all_files)}")
print(f"Posibles imágenes: {len(image_candidates)}")
print(f"Posibles máscaras: {len(mask_candidates)}")

pd.DataFrame({
    "path": [str(p) for p in all_files[:20]],
    "file_name": [p.name for p in all_files[:20]],
    "size_mb": [round(file_size_mb(p), 3) for p in all_files[:20]],
    "likely_mask": [is_likely_mask(p) for p in all_files[:20]],
})

## 6. Inspección rápida de estructura de carpetas

In [ ]:
folder_rows = []
for folder in sorted([p for p in DATASET_ROOT.rglob("*") if p.is_dir()])[:200]:
    files_in_folder = [p for p in folder.iterdir() if p.is_file()]
    medical_files = [p for p in files_in_folder if has_supported_suffix(p)]
    if files_in_folder:
        folder_rows.append({
            "folder": str(folder.relative_to(DATASET_ROOT)),
            "num_files": len(files_in_folder),
            "num_medical_files": len(medical_files),
            "examples": ", ".join([p.name for p in files_in_folder[:3]])
        })

folder_df = pd.DataFrame(folder_rows)
folder_df.head(30)

## 7. Funciones de lectura e inspección de imagen médica

In [ ]:
def read_medical_image(path: Path) -> tuple[sitk.Image, np.ndarray]:
    """Lee una imagen médica con SimpleITK y devuelve imagen SITK + array NumPy.

    SimpleITK devuelve arrays en orden [z, y, x] para 3D.
    """
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)
    return image, array

def inspect_image_header(path: Path) -> dict[str, Any]:
    """Inspecciona metadatos básicos sin cargar todo el volumen si el formato lo permite."""
    row: dict[str, Any] = {
        "path": str(path),
        "file_name": path.name,
        "relative_path": str(path.relative_to(DATASET_ROOT)) if path.is_relative_to(DATASET_ROOT) else str(path),
        "suffix": "".join(path.suffixes),
        "size_mb": round(file_size_mb(path), 3),
        "likely_mask": is_likely_mask(path),
        "read_ok": False,
        "error": "",
    }
    try:
        reader = sitk.ImageFileReader()
        reader.SetFileName(str(path))
        reader.ReadImageInformation()
        row.update({
            "read_ok": True,
            "dimension": reader.GetDimension(),
            "size": tuple(reader.GetSize()),
            "spacing": tuple(round(float(x), 5) for x in reader.GetSpacing()),
            "origin": tuple(round(float(x), 5) for x in reader.GetOrigin()),
            "pixel_type": reader.GetPixelIDTypeAsString(),
        })
    except Exception as exc:
        row["error"] = repr(exc)
    return row

def summarize_array(array: np.ndarray, is_mask: bool = False) -> dict[str, Any]:
    """Resumen básico de un array leído."""
    summary = {
        "array_shape": tuple(array.shape),
        "array_dtype": str(array.dtype),
        "min": float(np.nanmin(array)),
        "max": float(np.nanmax(array)),
        "mean": float(np.nanmean(array)),
    }
    if is_mask:
        unique_values = np.unique(array)
        summary["unique_values"] = unique_values[:30].tolist()
        summary["num_unique_values"] = int(len(unique_values))
        summary["nonzero_pixels"] = int(np.count_nonzero(array))
    return summary

## 8. Tabla resumen inicial del dataset

In [ ]:
MAX_FILES_TO_INSPECT = 500

header_rows = []
for path in all_files[:MAX_FILES_TO_INSPECT]:
    header_rows.append(inspect_image_header(path))

summary_df = pd.DataFrame(header_rows)
summary_csv_path = TABLES_DIR / "e3_e4_dataset_file_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print(f"Tabla resumen guardada en: {summary_csv_path}")
summary_df.head(20)

## 9. Selección de imagen y máscara de ejemplo

In [ ]:
IMAGE_PATH = None
MASK_PATH = None

def normalize_name_for_pairing(path: Path) -> str:
    name = path.name.lower()
    for suffix in SUPPORTED_SUFFIXES:
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    for keyword in MASK_KEYWORDS:
        name = name.replace(keyword, "")
    name = re.sub(r"[^a-z0-9]+", "", name)
    return name

def find_probable_pair(image_paths: list[Path], mask_paths: list[Path]) -> tuple[Path | None, Path | None]:
    if not image_paths:
        return None, None
    if not mask_paths:
        return image_paths[0], None

    for image_path in image_paths:
        image_key = normalize_name_for_pairing(image_path)
        for mask_path in mask_paths:
            mask_key = normalize_name_for_pairing(mask_path)
            if image_key and mask_key and (image_key in mask_key or mask_key in image_key):
                return image_path, mask_path
    return image_paths[0], mask_paths[0]

if IMAGE_PATH is None:
    selected_image_path, selected_mask_path = find_probable_pair(image_candidates, mask_candidates)
else:
    selected_image_path = Path(IMAGE_PATH)
    selected_mask_path = Path(MASK_PATH) if MASK_PATH is not None else None

print("Imagen seleccionada:", selected_image_path)
print("Máscara seleccionada:", selected_mask_path)

## 10. Lectura de imagen y máscara

In [ ]:
if selected_image_path is None:
    raise FileNotFoundError(
        f"No se encontraron imágenes en DATASET_ROOT={DATASET_ROOT}. "
        "Verificar ruta del dataset o extensiones soportadas."
    )

sitk_image, image_array = read_medical_image(selected_image_path)
image_info = {
    "path": str(selected_image_path),
    "sitk_dimension": sitk_image.GetDimension(),
    "sitk_size": tuple(sitk_image.GetSize()),
    "sitk_spacing": tuple(round(float(x), 5) for x in sitk_image.GetSpacing()),
    "sitk_origin": tuple(round(float(x), 5) for x in sitk_image.GetOrigin()),
    "sitk_direction": tuple(round(float(x), 5) for x in sitk_image.GetDirection()),
    **summarize_array(image_array, is_mask=False),
}
print(json.dumps(image_info, indent=2, ensure_ascii=False))

mask_array = None
sitk_mask = None
if selected_mask_path is not None:
    sitk_mask, mask_array = read_medical_image(selected_mask_path)
    mask_info = {
        "path": str(selected_mask_path),
        "sitk_dimension": sitk_mask.GetDimension(),
        "sitk_size": tuple(sitk_mask.GetSize()),
        "sitk_spacing": tuple(round(float(x), 5) for x in sitk_mask.GetSpacing()),
        "sitk_origin": tuple(round(float(x), 5) for x in sitk_mask.GetOrigin()),
        "sitk_direction": tuple(round(float(x), 5) for x in sitk_mask.GetDirection()),
        **summarize_array(mask_array, is_mask=True),
    }
    print("\\nMáscara:")
    print(json.dumps(mask_info, indent=2, ensure_ascii=False))
else:
    print("\\nNo se encontró máscara automáticamente. Se visualizará solo la imagen.")

## 11. Selección de corte para visualización

In [ ]:
def select_representative_slice(image: np.ndarray, mask: np.ndarray | None = None) -> int | None:
    if image.ndim < 3:
        return None
    if mask is not None and mask.ndim >= 3 and np.count_nonzero(mask) > 0:
        nonzero_per_slice = np.count_nonzero(mask, axis=tuple(range(1, mask.ndim)))
        return int(np.argmax(nonzero_per_slice))
    return int(image.shape[0] // 2)

def get_2d_slice(array: np.ndarray, slice_index: int | None) -> np.ndarray:
    if array.ndim == 2:
        return array
    if array.ndim == 3:
        if slice_index is None:
            slice_index = array.shape[0] // 2
        return array[slice_index]
    raise ValueError(f"No se soporta array con ndim={array.ndim} y shape={array.shape}")

slice_index = select_representative_slice(image_array, mask_array)
image_slice = get_2d_slice(image_array, slice_index)
mask_slice = get_2d_slice(mask_array, slice_index) if mask_array is not None else None

print("slice_index:", slice_index)
print("image_slice.shape:", image_slice.shape)
if mask_slice is not None:
    print("mask_slice.shape:", mask_slice.shape)
    print("mask unique values:", np.unique(mask_slice)[:30])

## 12. Visualización de imagen, máscara y superposición

In [ ]:
def normalize_for_display(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.float32)
    p1, p99 = np.percentile(image, [1, 99])
    image = np.clip(image, p1, p99)
    return (image - image.min()) / (image.max() - image.min() + 1e-8)

display_image = normalize_for_display(image_slice)

if mask_slice is None:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(display_image, cmap="gray")
    ax.set_title("Imagen seleccionada")
    ax.axis("off")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(display_image, cmap="gray")
    axes[0].set_title("Imagen original")
    axes[0].axis("off")

    axes[1].imshow(mask_slice, cmap="tab20")
    axes[1].set_title("Máscara")
    axes[1].axis("off")

    axes[2].imshow(display_image, cmap="gray")
    masked = np.ma.masked_where(mask_slice == 0, mask_slice)
    axes[2].imshow(masked, cmap="tab20", alpha=0.40)
    axes[2].set_title("Superposición imagen + máscara")
    axes[2].axis("off")
    fig.suptitle(f"Corte seleccionado: {slice_index}", fontsize=14)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
figure_path = FIGURES_DIR / f"e3_e4_overlay_{timestamp}.png"
fig.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"Figura guardada en: {figure_path}")

## 13. Registro de evidencia del caso seleccionado

In [ ]:
case_summary = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "image_path": str(selected_image_path),
    "mask_path": str(selected_mask_path) if selected_mask_path is not None else "",
    "figure_path": str(figure_path),
    "slice_index": slice_index,
    "image_shape": str(tuple(image_array.shape)),
    "image_dtype": str(image_array.dtype),
    "image_spacing": str(tuple(round(float(x), 5) for x in sitk_image.GetSpacing())),
    "image_size": str(tuple(sitk_image.GetSize())),
    "mask_shape": str(tuple(mask_array.shape)) if mask_array is not None else "",
    "mask_dtype": str(mask_array.dtype) if mask_array is not None else "",
    "mask_spacing": str(tuple(round(float(x), 5) for x in sitk_mask.GetSpacing())) if sitk_mask is not None else "",
    "mask_unique_values": str(np.unique(mask_array).tolist()[:50]) if mask_array is not None else "",
}

case_summary_df = pd.DataFrame([case_summary])
case_summary_csv_path = TABLES_DIR / f"e3_e4_selected_case_summary_{timestamp}.csv"
case_summary_df.to_csv(case_summary_csv_path, index=False)

print(f"Resumen del caso guardado en: {case_summary_csv_path}")
case_summary_df

## 14. Conclusión técnica inicial

In [ ]:
conclusion_lines = [
    "# Conclusión técnica E3/E4",
    "",
    f"- Fecha/hora: {datetime.now().isoformat(timespec='seconds')}",
    f"- Dataset root: `{DATASET_ROOT}`",
    f"- Archivos médicos encontrados: {len(all_files)}",
    f"- Posibles imágenes: {len(image_candidates)}",
    f"- Posibles máscaras: {len(mask_candidates)}",
    f"- Imagen seleccionada: `{selected_image_path}`",
    f"- Máscara seleccionada: `{selected_mask_path}`" if selected_mask_path else "- Máscara seleccionada: no encontrada automáticamente",
    f"- Shape imagen: `{tuple(image_array.shape)}`",
    f"- Spacing imagen: `{tuple(round(float(x), 5) for x in sitk_image.GetSpacing())}`",
    f"- Figura de evidencia: `{figure_path}`",
    f"- CSV resumen dataset: `{summary_csv_path}`",
    f"- CSV resumen caso: `{case_summary_csv_path}`",
    "",
    "## Observaciones",
    "",
    "- Este notebook verifica lectura, metadatos básicos y visualización.",
    "- No realiza diagnóstico ni interpretación clínica.",
    "- El emparejamiento automático imagen/máscara puede requerir ajuste según la estructura real del dataset.",
    "- Próximo paso recomendado: implementar un loader específico para SPIDER y documentar el mapeo de clases.",
]

conclusion_path = REPORTS_DIR / f"e3_e4_conclusion_{timestamp}.md"
conclusion_path.write_text("\n".join(conclusion_lines), encoding="utf-8")

print("\n".join(conclusion_lines))
print(f"\nConclusión guardada en: {conclusion_path}")

## 15. Próximos pasos

1. Confirmar estructura exacta del dataset principal.
2. Ajustar el loader específico para emparejar correctamente imagen y máscara.
3. Documentar clases disponibles en las máscaras.
4. Definir split por paciente si el dataset lo permite.
5. Preparar notebook siguiente para preprocesamiento reproducible y baseline de segmentación.